# Classical Portfolio Methods

This notebook implements and evaluates three classical portfolio optimization strategies for 13 Chinese pharmaceutical and biotechnology equities, using the [Riskfolio-Lib](https://riskfolio-lib.readthedocs.io/) library:

- **Mean–Variance Optimization (Sharpe Ratio Maximization):** Maximises the portfolio Sharpe ratio under a classical mean–variance framework.
- **Risk Parity (Equal Risk Contribution):** Allocates weights so each asset contributes equally to total portfolio variance.
- **Multi-Measure Optimization:** Tests Sharpe-optimal portfolios across alternative risk measures (CVaR, MAD, CDaR, etc.).

**Data:** `../data/Return.xlsx` — Daily returns, 13 China pharma/biotech stocks, 2018–2025

> **Note:** Results are backtest-based. This notebook is for research and educational purposes only and does not constitute investment advice.

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:.4%}'.format

# Load data
Y = pd.read_excel("../data/Return.xlsx")
Y['Date'] = pd.to_datetime(Y['Date'])
Y.set_index('Date', inplace=True)
Y.columns = Y.columns.str.strip()

print(Y.head())

2. Estimating Mean Variance Portfolios

2.1. Calculating the portfolio that maximizes Sharpe ratio.

In [ ]:
import riskfolio as rp
print(rp.__version__)



In [ ]:
# Building the portfolio object
port = rp.Portfolio(returns=Y)

# Calculating optimal portfolio

# Select method and estimate input parameters:

method_mu='hist' # Method to estimate expected returns based on historical data.
method_cov='hist' # Method to estimate covariance matrix based on historical data.

port.assets_stats(method_mu=method_mu, method_cov=method_cov)

# Estimate optimal portfolio:

model='Classic' # Could be Classic (historical), BL (Black Litterman) or FM (Factor Model)
rm = 'MV' # Risk measure used, this time will be variance
obj = 'Sharpe' # Objective function, could be MinRisk, MaxRet, Utility or Sharpe
hist = True # Use historical scenarios for risk measures that depend on scenarios
rf = 0 # Risk free rate
l = 0 # Risk aversion factor, only useful when obj is 'Utility'

w = port.optimization(model=model, rm=rm, obj=obj, rf=rf, l=l, hist=hist)

display(w.T)

2.2 Plotting portfolio composition

In [ ]:
# Plotting the composition of the portfolio

ax = rp.plot_pie(w=w, title='Sharpe Mean Variance', others=0, nrow=25, cmap = "tab20",
                 height=6, width=10, ax=None)

2.3 Plotting Risk Composition

In [ ]:
# Plotting the risk composition of the portfolio

ax = rp.plot_risk_con(w, cov=port.cov, returns=port.returns, rm=rm, rf=0, alpha=0.01,
                      color="tab:blue", height=6, width=10, ax=None)

3. Estimating Risk Parity Portfolios for Other Risk Measures

3.1 Calculating the risk parity portfolio for variance.

In [ ]:
b = None # Risk contribution constraints vector

w_rp = port.rp_optimization(model=model, rm=rm, rf=rf, b=b, hist=hist)

display(w_rp.T)

3.2 Plotting portfolio composition

In [ ]:
ax = rp.plot_pie(w=w_rp, title='Risk Parity Variance', others=0.05, nrow=25, cmap = "tab20",
                 height=6, width=10, ax=None)


3.3 Plotting Risk Composition

In [ ]:
ax = rp.plot_risk_con(w_rp, cov=port.cov, returns=port.returns, rm=rm, rf=0, alpha=0.01,
                      color="tab:blue", height=6, width=10, ax=None)

3.4 Calculate Optimal Portfolios for Several Risk Measures

In [ ]:
# 'MV': Standard Deviation.
# 'MAD': Mean Absolute Deviation.
# 'MSV': Semi Standard Deviation.
# 'FLPM': First Lower Partial Moment (Omega Ratio).
# 'SLPM': Second Lower Partial Moment (Sortino Ratio).


rms = ['MV', 'MAD', 'MSV', 'FLPM', 'SLPM', 'CVaR',
       'EVaR', 'CDaR', 'UCI', 'EDaR']
w_s = pd.DataFrame([])
success_rms = []

port.lowerret = 0
port.upperret = 1

for i in rms:
    try:
        w = port.optimization(
            model='Classic',
            rm=i,
            obj='Sharpe',
            rf=0,
            l=0,
            hist=True
        )
        w_s = pd.concat([w_s, w], axis=1)
        success_rms.append(i)
    except Exception as e:
        print(f"{i} failed: {e}")

w_s.columns = success_rms
w_s.style.format("{:.2%}").background_gradient(cmap='YlGn')


In [ ]:
import matplotlib.pyplot as plt

# Plotting a comparison of assets weights for each portfolio

fig = plt.gcf()
fig.set_figwidth(16)
fig.set_figheight(6)
ax = fig.subplots(nrows=1, ncols=1)

w_s.plot.bar(ax=ax)

In [ ]:
# Define end date as the latest available date
end_date = Y.index[-1]

# Define start date as 5 years before the end date
start_date = end_date - pd.DateOffset(years=5)

# Extract the 5-year data range
returns_5yr = Y.loc[start_date:end_date]

# Calculate cumulative return using the formula:
# R = (∏ (1 + r_i)) - 1
five_year_return = (1 + returns_5yr).prod() - 1

# Display the result in percentage format
print("5-Year Return for each asset:")
print(five_year_return.apply(lambda x: f"{x:.2%}"))


In [ ]:
# Define start date as 2 year before the end date
start_date_2yr = end_date - pd.DateOffset(years=2)

# Extract the 2-year data range
returns_2yr = Y.loc[start_date_2yr:end_date]

# Calculate cumulative 1-year return using the same formula:
# R = (∏ (1 + r_i)) - 1
two_year_return = (1 + returns_2yr).prod() - 1

# Display the result in percentage format
print("\n2-Year Return for each asset:")
print(two_year_return.apply(lambda x: f"{x:.2%}"))